# 03 — Il confronto (tentativo 12: solo politici, più testate, etichette NewsGuard 2026)

Sulle testate valutate del tentativo 12, fold per gruppo, per testata: **feature** (TF-IDF + LR, voto per testata) e
**ModernBERT-base** (seme 42, media delle probabilità), con le baseline. Poi l'effetto delle **testate in più**: sulle
testate valutate sia qui sia nel tentativo 11 (variante "politici", 143 testate), i modelli di qui (addestrati su più
testate) contro quelli del tentativo 11 (`Dataset/riferimento_t11/`), sugli stessi ricampionamenti.

Metriche: F1 macro sui 5 livelli, accuratezza, ±1 livello, binario (punteggio ≥ 60). Intervalli al 95% con bootstrap
per gruppo (2000 ricampionamenti).

In [1]:
import json

import numpy as np
import pandas as pd

from comune import CLASSI, LIVELLI, RESULTS, VERSIONI, metriche_di, per_testata

B = 2000
OUT = RESULTS / "confronto"; OUT.mkdir(parents=True, exist_ok=True)
RIF = VERSIONI.parent / "riferimento_t11"
t = pd.read_csv(VERSIONI / "fold.csv").set_index("domain")
val = t[t.valutazione].sort_index()
METR = ["f1_macro", "accuratezza", "tolerant_1", "bin_accuratezza"]


def quattro(y, p):
    M = np.bincount(y * 5 + p, minlength=25).reshape(5, 5)
    tp = np.diag(M); den = M.sum(0) + M.sum(1)
    f1 = np.divide(2 * tp, den, out=np.zeros(5), where=den > 0).mean()
    d = np.abs(y - p)
    return [f1, (d == 0).mean(), (d <= 1).mean(), ((y >= 2) == (p >= 2)).mean()]


def leggi(cartella, prefisso):
    P = {}
    f = cartella / f"{prefisso}_fold_gruppo_articoli.csv"
    if f.exists():
        P["feature"] = per_testata(pd.read_csv(f)).y_pred
    f = cartella / "transformer" / prefisso.replace("feature_tuned", "modernbert") / "articoli.csv"
    if not f.exists():
        f = cartella / "transformer" / (prefisso.replace("feature_tuned", "modernbert") + "_s42") / "articoli.csv"
    if (f.parent / "FATTO").exists():
        p = pd.read_csv(f).groupby("domain")[CLASSI].mean()
        P["ModernBERT"] = pd.Series(p.to_numpy().argmax(1), index=p.index)
    return P


QUI = leggi(RESULTS, "feature_tuned_2026")
T11 = leggi(RIF, "feature_tuned_politici_2026")
assert QUI, "nessun modello finito"
print("modelli di qui:", list(QUI), "| del tentativo 11:", list(T11))

modelli di qui: ['feature', 'ModernBERT'] | del tentativo 11: ['feature', 'ModernBERT']


In [2]:
def campioni(idx):
    g = val.gruppo[idx].to_numpy(); gruppi = np.unique(g); pos = {x: np.where(g == x)[0] for x in gruppi}
    rng = np.random.default_rng(0)
    return [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def misura(pred, idx, camp):
    y = val.label[idx].to_numpy().astype(int); p = pred.reindex(idx).to_numpy().astype(int)
    m = metriche_di(y, p); assert np.allclose([m[k] for k in METR], quattro(y, p))
    return m, np.array([quattro(y[i], p[i]) for i in camp])


tutte = val.index; camp_tutte = campioni(tutte)
righe, BS = [], {}
for mod, pred in QUI.items():
    m, b = misura(pred, tutte, camp_tutte); BS[mod] = b
    righe.append({"modello": mod, "testate": f"valutate qui ({len(tutte)})", **{k: m[k] for k in METR},
                  **{f"{k} 2.5%": np.percentile(b[:, j], 2.5) for j, k in enumerate(METR)}, **{f"{k} 97.5%": np.percentile(b[:, j], 97.5) for j, k in enumerate(METR)}})
for nome in ["maggioranza", "a_caso"]:
    f = RESULTS / f"baseline_{nome}_2026_fold_gruppo_metriche.json"
    if f.exists():
        m = json.load(open(f))["testata_voto"]; righe.append({"modello": f"baseline: {nome.replace('_', ' ')}", "testate": f"valutate qui ({len(tutte)})", **{k: m[k] for k in METR}})
diff = []
if {"feature", "ModernBERT"} <= set(QUI):
    d = BS["ModernBERT"] - BS["feature"]; a, b = metriche_di(val.label.to_numpy().astype(int), QUI["ModernBERT"].reindex(tutte).to_numpy().astype(int)), metriche_di(val.label.to_numpy().astype(int), QUI["feature"].reindex(tutte).to_numpy().astype(int))
    diff.append({"confronto": "ModernBERT − feature", "testate": f"valutate qui ({len(tutte)})", **{f"Δ {k}": a[k] - b[k] for k in METR},
                 **{f"Δ {k} 2.5%": np.percentile(d[:, j], 2.5) for j, k in enumerate(METR)}, **{f"Δ {k} 97.5%": np.percentile(d[:, j], 97.5) for j, k in enumerate(METR)}})
# l'effetto delle testate in più: sulle testate valutate qui e nel tentativo 11, con le stesse etichette 2026
comuni = tutte[tutte.isin(pd.read_csv(RIF / "solo_politici.csv").query("valutazione").domain)]
comuni = comuni[[all(d in P[m].index for P in (QUI, T11) for m in P) for d in comuni]]
print(f"testate valutate qui e nel tentativo 11: {len(comuni)}")
if len(comuni) >= 30:
    camp_c = campioni(comuni)
    for mod in sorted(set(QUI) & set(T11)):
        (ma, ba), (mb, bb) = misura(QUI[mod], comuni, camp_c), misura(T11[mod], comuni, camp_c)
        for nome, m, b in [(f"{mod} (tentativo 12, più testate)", ma, ba), (f"{mod} (tentativo 11)", mb, bb)]:
            righe.append({"modello": nome, "testate": f"in comune con il tentativo 11 ({len(comuni)})", **{k: m[k] for k in METR},
                          **{f"{k} 2.5%": np.percentile(b[:, j], 2.5) for j, k in enumerate(METR)}, **{f"{k} 97.5%": np.percentile(b[:, j], 97.5) for j, k in enumerate(METR)}})
        d = ba - bb
        diff.append({"confronto": f"{mod}: più testate (12) − tentativo 11", "testate": f"in comune ({len(comuni)})", **{f"Δ {k}": ma[k] - mb[k] for k in METR},
                     **{f"Δ {k} 2.5%": np.percentile(d[:, j], 2.5) for j, k in enumerate(METR)}, **{f"Δ {k} 97.5%": np.percentile(d[:, j], 97.5) for j, k in enumerate(METR)}})
tab = pd.DataFrame(righe); tab.to_csv(OUT / "intervalli.csv", index=False)
diff = pd.DataFrame(diff); diff.to_csv(OUT / "differenze.csv", index=False)
(OUT / "aggiornato.txt").write_text(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))
display(tab[["modello", "testate"] + METR].round(3)); diff.round(3)

testate valutate qui e nel tentativo 11: 133


,modello,testate,f1_macro,accuratezza,tolerant_1,bin_accuratezza
0,feature,valutate qui (254),0.383,0.441,0.768,0.697
1,ModernBERT,valutate qui (254),0.312,0.346,0.689,0.638
2,baseline: maggioranza,valutate qui (254),0.098,0.323,0.602,0.602
3,baseline: a caso,valutate qui (254),0.184,0.213,0.547,0.528
4,"ModernBERT (tentativo 12, più testate)",in comune con il tentativo 11 (133),0.280,0.293,0.684,0.639
5,ModernBERT (tentativo 11),in comune con il tentativo 11 (133),0.271,0.286,0.714,0.579
6,"feature (tentativo 12, più testate)",in comune con il tentativo 11 (133),0.320,0.376,0.729,0.632
7,feature (tentativo 11),in comune con il tentativo 11 (133),0.279,0.338,0.752,0.639


,confronto,testate,Δ f1_macro,Δ accuratezza,Δ tolerant_1,Δ bin_accuratezza,Δ f1_macro 2.5%,Δ accuratezza 2.5%,Δ tolerant_1 2.5%,Δ bin_accuratezza 2.5%,Δ f1_macro 97.5%,Δ accuratezza 97.5%,Δ tolerant_1 97.5%,Δ bin_accuratezza 97.5%
0,ModernBERT − feature,valutate qui (254),-0.071,-0.094,-0.079,-0.059,-0.148,-0.177,-0.145,-0.143,0.008,-0.012,-0.012,0.024
1,ModernBERT: più testate (12) − tentativo 11,in comune (133),0.009,0.008,-0.030,0.060,-0.109,-0.106,-0.114,-0.030,0.118,0.119,0.051,0.151
2,feature: più testate (12) − tentativo 11,in comune (133),0.040,0.038,-0.023,-0.008,-0.035,-0.038,-0.094,-0.073,0.117,0.119,0.054,0.060
